In [ ]:
!git clone https://github.com/NASA-IMPACT/Surya.git
%cd Surya
!pip install -q -e .
!pip install -q h5netcdf huggingface_hub awscli

In [ ]:
import torch, os, sys, shutil, time, gc
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import yaml

print('CUDA available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

sys.path.insert(0, 'easy_inference')
import run_easy_inference as ezi

In [ ]:
from huggingface_hub import snapshot_download

with open('easy_inference/config_easy.yaml') as f:
    full_cfg = yaml.safe_load(f)
advanced = full_cfg['advanced']

model_dir = Path('data/Surya-1.0')
model_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id=advanced['model_repo_id'], local_dir=str(model_dir), allow_patterns=list(advanced['model_allow_patterns']), token=None)

with open(advanced['foundation_config_path']) as f:
    base_config = yaml.safe_load(f)
base_config['data']['time_delta_input_minutes'] = advanced['time_delta_input_minutes']
base_config['data']['n_input_timestamps'] = len(advanced['time_delta_input_minutes'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ezi.build_model(base_config)
state = torch.load(advanced['weights_path'], map_location='cpu')
model.load_state_dict(state if not isinstance(state, dict) or 'state_dict' not in state else state['state_dict'])
model.eval()
model.to(device)
model.finetune = True

scalers = ezi.build_scalers(yaml.safe_load(open(advanced['scalers_path'])))
channels = base_config['data']['sdo_channels']
pooling = base_config['data'].get('pooling')
print('model ready, finetune =', model.finetune, '| device =', device)

In [ ]:
def extract_embedding(reference_ts, workdir=Path('/kaggle/temp')):
    input_offsets = sorted(advanced['time_delta_input_minutes'])
    start_dt = reference_ts + timedelta(minutes=input_offsets[0])
    end_dt = reference_ts + timedelta(minutes=input_offsets[-1])
    tag = reference_ts.strftime('%Y%m%d_%H%M')
    val_dir = workdir / f'sdo_{tag}'
    idx_path = workdir / f'index_{tag}.csv'
    ezi.download_surya_bench_range(bucket=advanced['s3_bucket'], output_dir=val_dir, start_datetime=start_dt, end_datetime=end_dt, cadence_minutes=advanced['cadence_minutes'], skip_existing=True, verify_size=False, match_tolerance_minutes=advanced['download_match_tolerance_minutes'], prune_to_expected=False, show_progress=False)
    ezi.build_index_csv_for_range(validation_data_dir=val_dir, index_path=idx_path, start_datetime=start_dt, end_datetime=end_dt, cadence_minutes=advanced['cadence_minutes'])
    idx_df = pd.read_csv(idx_path)
    idx_df['timestep'] = pd.to_datetime(idx_df['timestep'])
    present_index = idx_df.set_index('timestep')
    dataset = ezi.InputOnlyRolloutDataset(present_index=present_index, reference_timestamps=[reference_ts], channels=channels, time_delta_input_minutes=advanced['time_delta_input_minutes'], time_delta_target_minutes=advanced['time_delta_target_minutes'], prediction_steps=1, scalers=scalers, pooling=pooling, debug_logger=None)
    loader = ezi._build_single_sample_dataloader(dataset, num_workers=0, prefetch_factor=None, pin_memory=False)
    batch = next(iter(loader))
    model_input = {k: v.to(device) for k, v in batch[0].items() if torch.is_tensor(v)}
    with torch.no_grad(): tokens = model(model_input)
    pooled = tokens.mean(dim=1).squeeze(0).cpu().numpy()
    del batch, model_input, tokens, dataset, loader
    gc.collect()
    shutil.rmtree(val_dir, ignore_errors=True)
    if val_dir.exists():
        print('WARNING: cleanup failed for', val_dir)
    idx_path.unlink(missing_ok=True)
    return pooled

In [ ]:
calibration_dates = [pd.Timestamp('2011-06-15 12:00:00'), pd.Timestamp('2018-06-15 12:00:00'), pd.Timestamp('2024-06-15 12:00:00')]
for ts in calibration_dates:
    t0 = time.time()
    try:
        vec = extract_embedding(ts)
        print(ts, 'OK shape', vec.shape, 'elapsed', round(time.time() - t0, 1), 's')
    except Exception as e:
        print(ts, 'FAILED:', repr(e), 'elapsed', round(time.time() - t0, 1), 's')

In [ ]:
output_path = Path('/kaggle/working/embeddings.npz')
checkpoint_path = Path('/kaggle/input/heliomag-embeddings-checkpoint/embeddings_merged.npz')
start = pd.Timestamp('2010-05-16 12:00:00')
end = pd.Timestamp('2021-12-31 12:00:00')  # phase 3: densify training window to every-3-days
timestamps = pd.date_range(start, end, freq='3D')
print('total timestamps:', len(timestamps), flush=True)
existing = dict(np.load(checkpoint_path, allow_pickle=True)) if checkpoint_path.exists() else {}
results = existing
print('already done:', len(results), flush=True)
t_start = time.time()
for i, ts in enumerate(timestamps):
    key = ts.strftime('%Y%m%d')
    if key in results: continue
    try:
        results[key] = extract_embedding(ts)
    except Exception as e:
        print(i, ts, 'FAILED', repr(e), flush=True)
        continue
    if len(results) % 10 == 0:
        np.savez(output_path, **results)
        du = lambda p: round(shutil.disk_usage(p).free / 1e9, 1)
        print(i, len(timestamps), key, 'done=', len(results), 'elapsed_min=', round((time.time() - t_start) / 60, 1), 'free_gb_temp=', du('/kaggle/temp'), 'free_gb_root=', du('/'), 'free_gb_working=', du('/kaggle/working'), flush=True)
    if len(results) % 20 == 0:
        shutil.rmtree('/kaggle/temp', ignore_errors=True)
        Path('/kaggle/temp').mkdir(exist_ok=True)
np.savez(output_path, **results)
print('FINAL saved', len(results), 'embeddings to', output_path, flush=True)


In [ ]:
import os, numpy as np
p = '/kaggle/working/embeddings.npz'
print('exists:', os.path.exists(p))
print('size bytes:', os.path.getsize(p) if os.path.exists(p) else 0)
print('n embeddings:', len(np.load(p)) if os.path.exists(p) else 0)